<a href="https://colab.research.google.com/github/jshogland/SpatialModelingTutorials/blob/main/Notebooks/GroupingPlotsBasedOnHilbertTrees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unsupervised clustering of similar FIA plots based on tree data.
### This approach uses Hilbert Curves to transform FIA tree table values (continuous and categorical) into encoded values that can be used to cluster similar plots, based on the composition of plot/subplot tree measurements.

Created by John Hogland 10/12/2026

## Install packages

In [ ]:
!pip install hilbertcurve

## Import libraries

In [ ]:
import sqlite3, pandas as pd, numpy as np, gdown, zipfile, os, geopandas as gpd

## Download FIA data for our study area (Indiana)

In [ ]:
url=r'https://apps.fs.usda.gov/fia/datamart/Databases/SQLite_FIADB_IN.zip'
out_zip_fl= 'fiadb.zip'

if(not os.path.exists(out_zip_fl)):
    gdown.download(url=url,output=out_zip_fl,quiet=False,fuzzy=True)
    with zipfile.ZipFile(out_zip_fl, 'r') as zip_ref:
        zip_ref.extractall(".")





## Get the PLOT and TREE table

In [ ]:
def get_tables(db_path,tbl_lst):
  '''
  Returns all records as a dataframe from specified tables as a list of dataframes

  db_path = (str) path to sqlight database
  tbl_lst = (list) table names as strings

  returns list of dataframes in the same order as specified tbl names

  '''
  conn=sqlite3.connect(db_path)
  cursor = conn.cursor()
  dbf_lst=[]
  for tbl in tbl_lst:
    sql="SELECT * FROM " + tbl
    dbf_lst.append(pd.read_sql(sql,conn))

  conn.close()
  return dbf_lst

plot,tree=get_tables('SQLite_FIADB_IN.db',['PLOT','TREE'])

## Convert dataframes to geodataframes
- Get the last 10 years of inventory and remove duplicate geometries (keeping the latest)

In [ ]:
plot_df=gpd.GeoDataFrame(plot,geometry=gpd.GeoSeries.from_xy(plot.LON,plot.LAT,crs='EPSG:4326'))
plot_df=plot_df[plot_df.INVYR>=2015].sort_values(by='INVYR') #get only plots form 10 years ago
print(plot_df.shape)
plot_df2=plot_df.drop_duplicates('geometry',keep='last')
print(plot_df2.shape)


## Subset the tree table to just the measured tree values, group on PLT_CN. Convert each plot's tree data into a standardized 2d array, and flatten the 2d array into a 1d array. Doing this will allow us to use the trees in each plot as an encoding which can further be converted to an embedding of similarity.

In [ ]:
tree2=(tree[['PLT_CN','PLOT','SUBP','TREE','STATUSCD','SPCD','DIA','HT']].loc[tree.PLT_CN.isin(plot_df2.CN)]).fillna(0)
grp=tree2.groupby('PLT_CN')
fsize=grp.count().max().max()
out_dic={}
for idx, group in grp:
    tarr=group.to_numpy()[:,2:]
    utarr=np.zeros((fsize,tree2.shape[1]-2))
    utarr[:tarr.shape[0],:]=tarr
    out_dic[idx]=utarr.ravel()


### Get just the standardized 1d tree values (X)

In [ ]:
tdf=pd.DataFrame.from_dict(out_dic,orient='index')
X=tdf.values
y=tdf.index

## Create Hilbert curve with precision 14 bits and 450 dimensions
### six fields * 75 trees (max number of trees) = 450 dimensions

calculate hilbert distance and merger values back to plots table

In [ ]:
from hilbertcurve.hilbertcurve import HilbertCurve

#create Hilbert Curve
hc2=HilbertCurve(14,450) #14 bit to capture species codes adn 450 columns based on fsize of the plot with the largest number of trees
dist=hc2.distances_from_points(X.astype('uint32')) #get distances note converting all floats to integers
hdf=pd.DataFrame({'plt_cn':y,'hdist':dist},dtype='object') #create dataframe
#merge distances to plots
plot_dist=plot_df2.merge(hdf,left_on='CN',right_on='plt_cn',how='left').drop(columns=['plt_cn'])
plot_dist

## Now that we have the hypercube defined and trained with numbers we can use any Hilbert distance to decode the trees on a given plot. Let's make up a Hilbert distance and decode it to view the trees for that distance.


In [ ]:
#create a new hilbert distance number from an existing number
nn=dist[200]-650000000120000000000000012378963852

#decode the distane to tree values
vls=hc2.point_from_distance(nn)

#convert those values to a dataframe formated similarly to the tree table
ntree=pd.DataFrame(np.array(vls).reshape(75,6),columns=[tree2.columns[-6:]])
ntree=ntree[(ntree.SUBP.values>0) & (ntree.SPCD.values>0) & (ntree.TREE.values > 0)].reset_index(drop=True) #do some cleaning to remove imposible records

#Display the table
ntree

## In addition to coding and decoding distances we can use distances to group similar plots based on the trees. There are a few ways to acomplish grouping similar plots. In this case we will use bit math to reduce the grainularity of the hilbert curve and naturally group like plots.

Given the dimensionality of the data, the size of the hilber distances, and common python computing packages, we can't use one dimensioinal clustering techniques. However, we can use bit math to reduce the size of the hilbert distances and naturally group plots as follows:
- Itteratively shift bits until we have approximately 10 groups
- Apply reduction to distances
- Remap new distances to classes

Find bit shift for 10 classes

In [ ]:
#convert distance list into an array
d_arr=np.array(dist)
#setup iterations to determine how reduction
nc=d_arr[0]
b=1
while nc >= 10:
  vls=np.unique(d_arr>>b)
  nc=vls.shape[0] #number of unique classes
  b+=1 #incrament the amount to shift the bit order

print('Shift by:',b)


Remap values

In [ ]:
hdist=plot_dist.hdist.copy()
hdist.loc[hdist.isna()]=-1
nvls=hdist.values >> b #shift
cvls=np.zeros(nvls.shape)
uvls=np.unique(nvls)
cnt=0
for v in uvls:
  cvls[nvls==v]=cnt
  cnt+=1

plot_dist['label']=cvls

## Visualize plot labels

In [ ]:
plot_dist.explore(column='label',cmap='terrain',legend=True,tiles='CyclOSM')